In [3]:
import pandas as pd
import os
import numpy as np
from nibabel.testing import data_path
import nibabel as nib

data = pd.read_csv("./dataset/meta.csv", sep=";")
# data.value_counts("study_type")


In [ ]:
# Load femur masks and check data
# dataset/s1379/segmentations/femur_left.nii.gz
# dataset/s1379/segmentations/femur_right.nii.gz
# dataset/s1379/segmentations/hip_left.nii.gz

import csv
import scipy

def getOrientation(image):
    x, y, z = nib.aff2axcodes(image.affine)

    if not (x == "R" and y == "A" and z == "S"):
        return "Wrong Orientation"

def checkIntensities(image): 
    intensity = image.get_fdata(dtype=np.float32)
    print("\nmax value: ", intensity.max())
    print("\nmin value: ", intensity.min())

    # Check that the volumes of the left and right are similar
    nib.imagestats.mask_volume(image)

def checkMetal(hu_values, mask):
    # Dilate the mask so an implant just outside the bone is caught too
    expanded_mask = scipy.ndimage.binary_dilation(mask)

    # Count the HU values of 2500 or more inside the dilated mask
    return int((hu_values[expanded_mask] >= 2500).sum())
    

def checkMaskMatchesAffines(ct, image):
    # Does the mask have the same shape and affine as the CT?
    if ct.shape != image.shape or not np.allclose(ct.affine, image.affine):
        return "Mask shape/affine doesn't match the CT"
    
def checkSlices(ct, mask):
    total_slices = ct.shape[2] # Number of total slices
    mask_slices = np.where(np.any(mask, axis=(0, 1)))[0] # Which slices does the mask touch?

    return bool(total_slices - 1 <= mask_slices[-1])

def checkBoneLength(image, mask):
    mask_slices = np.where(np.any(mask, axis=(0, 1)))[0] # Which slices does the mask touch?
    voxel_size = image.header.get_zooms()[2] 

    return float(len(mask_slices) * voxel_size)


def loadData(label): # Load a single CT with both femurs
    ct = nib.load(f"dataset/{label}/ct.nii.gz")
    femurLeft = nib.load(f"dataset/{label}/segmentations/femur_left.nii.gz")
    femurRight = nib.load(f"dataset/{label}/segmentations/femur_right.nii.gz")

    return ct, {"left": femurLeft, "right": femurRight}


def checkDataForProblems():
    rows = []

    for label, study_type in zip(data.image_id, data.study_type):
        ct, femurs = loadData(label)
        hu_values = ct.get_fdata(dtype=np.float32) # Read the CT once per scan

        for side, image in femurs.items():
            raw = np.asarray(image.dataobj) # Read the mask once
            mask = raw > 0
            row = {"subject_id": label, "study_type": study_type, "side": side,
                   "has_femur": bool(mask.any()),
                   "femur_voxels": int(mask.sum()),
                   "volume_ml": float(mask.sum() * np.prod(image.header.get_zooms()) / 1000),
                   "superior_cut": None, "extent_from_top_mm": None, "metal_voxels": None}

            reasons = [checkMaskMatchesAffines(ct, image), getOrientation(image)]
            if not ((raw == 0) | (raw == 1)).all():
                reasons.append("Mask values are not only 0 and 1")

            if row["has_femur"]:
                row["superior_cut"] = checkSlices(ct, mask)
                row["extent_from_top_mm"] = checkBoneLength(image, mask)
                row["metal_voxels"] = checkMetal(hu_values, mask) # apparently expands the mask somewhat, so the metal check can catch an implant just outside the bone
                if row["superior_cut"]: reasons.append("Mask touches the top slice")
                if row["extent_from_top_mm"] < 100: reasons.append("Less than 100mm of bone")
                if row["metal_voxels"] >= 50: reasons.append("Metal detected")

            else:
                reasons.append("Femur is empty")

            reasons = [r for r in reasons if r] # Drop the Nones
            row["include"] = len(reasons) == 0
            row["excluded_reason"] = "; ".join(reasons)
            rows.append(row)
            print(label, side, row["excluded_reason"] or "OK")

    results = pd.DataFrame(rows)
    os.makedirs("outputs", exist_ok=True)
    results.to_csv("outputs/femur_data.csv", index=False)
    return results


In [ ]:
results = checkDataForProblems()

withFemur = results[results.has_femur]
headComplete = withFemur[(withFemur.superior_cut == False) & (withFemur.extent_from_top_mm >= 100)]
noMetal = headComplete[headComplete.metal_voxels < 50]

print("Scans:", results.subject_id.nunique(), "-> femur sides:", len(results))
print("With femur:", len(withFemur))
print("Head complete:", len(headComplete))
print("No metal:", len(noMetal))
print("Usable sides:", results.include.sum(), "from", results[results.include].subject_id.nunique(), "scans")

In [ ]:
# Left/right volume check: scans where one femur is more than 25% bigger than the other
volumes = withFemur.pivot(index="subject_id", columns="side", values="volume_ml").dropna()
ratio = volumes.left / volumes.right
print(volumes[(ratio < 0.8) | (ratio > 1.25)])